# 🔧 Notebook 02 — Feature Engineering

Demonstrates each transformation step with before/after comparisons.

**Contents**
1. Load cleaned data
2. Engineered features walkthrough
3. Encoding pipeline
4. Scaling demonstration
5. Final feature matrix summary

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from data_preprocessing import load_data, clean_data, encode_features, scale_features, split_data
from feature_engineering import engineer_features

In [ ]:
RAW = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df_clean = clean_data(load_data(RAW))
print(f'Shape after cleaning: {df_clean.shape}')
df_clean[['tenure','MonthlyCharges','TotalCharges']].describe()

## 2. Engineered Features Walkthrough

In [ ]:
# Feature: AverageChargePerMonth
print('=== AverageChargePerMonth ===')
sample = df_clean[['tenure','MonthlyCharges','TotalCharges']].head(5).copy()
sample['AverageChargePerMonth'] = np.where(
    sample['tenure'] > 0,
    sample['TotalCharges'] / sample['tenure'],
    sample['MonthlyCharges']
).round(2)
sample

In [ ]:
# Feature: TenureGroup
print('=== TenureGroup distribution ===')
bins = [0, 12, 36, 100]
labels = ['New (0-12m)', 'Medium (13-36m)', 'Long (36m+)']
df_clean['TenureGroup_preview'] = pd.cut(df_clean['tenure'], bins=bins, labels=labels, include_lowest=True)

group_churn = df_clean.groupby('TenureGroup_preview', observed=True)['Churn'].agg(['mean','count'])
group_churn.columns = ['Churn Rate', 'Count']
group_churn['Churn Rate'] = group_churn['Churn Rate'].mul(100).round(1).astype(str) + '%'

fig, ax = plt.subplots(figsize=(8,4))
df_clean['TenureGroup_preview'].value_counts().sort_index().plot(kind='bar', ax=ax, color=['#dc2626','#d97706','#16a34a'], edgecolor='white')
ax.set_title('Customer Distribution by Tenure Group', fontweight='bold')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
df_clean.drop(columns=['TenureGroup_preview'], inplace=True)
group_churn

In [ ]:
# Run full feature engineering
df_feat = engineer_features(df_clean)
new_cols = [c for c in df_feat.columns if c not in df_clean.columns]
print(f'New features added: {new_cols}')
df_feat[new_cols + ['Churn']].head(8)

In [ ]:
# Correlation of new features with Churn
print('=== New feature correlation with Churn ===')
numeric_df = df_feat[new_cols + ['Churn']].select_dtypes(include='number')
corr_with_churn = numeric_df.corr()['Churn'].drop('Churn').sort_values(key=abs, ascending=False)
print(corr_with_churn.to_string())

## 3. Encoding Pipeline

In [ ]:
df_encoded, meta = encode_features(df_feat)
print(f'Shape before encoding: {df_feat.shape}')
print(f'Shape after encoding:  {df_encoded.shape}')
print(f'\nOne-hot encoded columns: {meta["one_hot_encoded"]}')
df_encoded.head(3)

In [ ]:
print('All feature columns:')
for i, col in enumerate(df_encoded.columns, 1):
    print(f'  {i:>3}. {col}')

## 4. Scaling Demonstration

In [ ]:
X_train, X_test, y_train, y_test = split_data(df_encoded)
X_train_s, X_test_s, scaler = scale_features(X_train, X_test)

scale_cols = ['tenure','MonthlyCharges','TotalCharges','AverageChargePerMonth']
scale_cols = [c for c in scale_cols if c in X_train_s.columns]

print('=== Before Scaling ===')
print(X_train[scale_cols].describe().loc[['mean','std','min','max']].round(2))

print('\n=== After Scaling (StandardScaler) ===')
print(X_train_s[scale_cols].describe().loc[['mean','std','min','max']].round(4))

## 5. Final Feature Matrix Summary

In [ ]:
print(f'Training set:  {X_train_s.shape[0]:,} rows × {X_train_s.shape[1]} features')
print(f'Test set:      {X_test_s.shape[0]:,} rows × {X_test_s.shape[1]} features')
print(f'Class balance (train): {y_train.mean():.2%} churn')
print(f'Class balance (test):  {y_test.mean():.2%} churn')

# Check for any remaining NaNs
nan_count = X_train_s.isna().sum().sum()
print(f'\nNaN values in training set: {nan_count}')